# OmniVoice FP16 Export to Google Drive

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fauziardiantama/aichan-extension/blob/main/OmniVoice.ipynb)

Notebook ini melakukan konversi dan ekspor bobot model **OmniVoice** ke presisi `torch.float16` secara bersih (tanpa duplikasi `audio_tokenizer`), kemudian menyimpannya ke folder `aichan-extension-files/omnivoice-float16` di Google Drive.

## 1. Install Dependencies

In [ ]:
!pip install -q omnivoice

## 2. Mount Google Drive & Siapkan Direktori Output

In [ ]:
from google.colab import drive
import os

# Mount Google Drive
drive.mount('/content/drive')

# Target folder di Google Drive
base_dir = '/content/drive/MyDrive/aichan-extension-files'
output_dir = os.path.join(base_dir, 'omnivoice-float16')
audio_tok_dir = os.path.join(output_dir, 'audio_tokenizer')

os.makedirs(audio_tok_dir, exist_ok=True)
print(f"Target output directory: {output_dir}")

## 3. Load & Export OmniVoice dalam Presisi FP16 (Proper Separation)

In [ ]:
import torch
from omnivoice import OmniVoice

print("Memuat model OmniVoice dari Hugging Face...")
# Gunakan cuda jika tersedia, atau cpu
device = "cuda:0" if torch.cuda.is_available() else "cpu"

model = OmniVoice.from_pretrained(
    "k2-fsa/OmniVoice",
    device_map=device,
    dtype=torch.float16,
)

# 1. Konversi dan simpan audio tokenizer ke FP16 di subfolder sendiri
audio_tok = model.audio_tokenizer
if audio_tok is not None:
    print("Mengonversi audio tokenizer ke FP16...")
    audio_tok.to(torch.float16)
    print(f"Menyimpan audio tokenizer FP16 ke: {audio_tok_dir}")
    audio_tok.save_pretrained(audio_tok_dir)
    # Lepas sementara agar TIDAK ikut terseret ke dalam state_dict model.safetensors
    model.audio_tokenizer = None

if model.feature_extractor is not None:
    print(f"Menyimpan feature extractor ke: {audio_tok_dir}")
    model.feature_extractor.save_pretrained(audio_tok_dir)

# 2. Simpan backbone model (murni backbone DiT/LLM FP16 ~1.17 GB tanpa duplikasi)
print(f"Menyimpan backbone model FP16 ke: {output_dir}...")
model.save_pretrained(output_dir)

# 3. Simpan text tokenizer
if model.text_tokenizer is not None:
    print(f"Menyimpan text tokenizer ke: {output_dir}...")
    model.text_tokenizer.save_pretrained(output_dir)

print("\nKonversi dan penyimpanan FP16 berhasil selesai!")

## 4. Verifikasi Ukuran File di Google Drive

In [ ]:
total_bytes = 0
print(f"{'File':<50} {'Ukuran (MB)':>12}")
print("-" * 65)

for root, _, files in os.walk(output_dir):
    for f in sorted(files):
        fp = os.path.join(root, f)
        sz = os.path.getsize(fp)
        total_bytes += sz
        rel_path = os.path.relpath(fp, output_dir)
        print(f"{rel_path:<50} {sz / (1024 * 1024):>12.2f}")

print("-" * 65)
total_mb = total_bytes / (1024 * 1024)
total_gb = total_bytes / (1024 * 1024 * 1024)
print(f"{'TOTAL':<50} {total_mb:>12.2f} MB ({total_gb:.2f} GB)")